# 🏥 Triage DPO Trainer (Google Colab Edition)

**Runtime required: T4 GPU** → Runtime → Change runtime type → T4 GPU

This notebook:
1. Installs HuggingFace stack **without touching PyTorch** (avoids CUDA mismatch)
2. Downloads Kaggle medical datasets and builds 6,000+ DPO training pairs
3. Fine-tunes `Qwen2.5-1.5B-Instruct` using LoRA + DPO on the free T4 GPU
4. Saves the trained adapter to Google Drive automatically

In [ ]:
# ✅ FIX: Do NOT uninstall or reinstall torch/torchvision.
# Colab pre-installs PyTorch compiled for the exact CUDA version on the runtime.
# Only install the HuggingFace ML stack on top.

import subprocess, sys

pkgs = [
    "kagglehub==0.3.4",
    "datasets==2.21.0",
    "transformers==4.46.3",
    "tokenizers==0.20.3",
    "accelerate==0.34.2",
    "peft==0.13.2",
    "trl==0.11.4",
    "bitsandbytes>=0.43.0",
    "huggingface-hub>=0.26.0",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"
] + pkgs)

# Verify CUDA is working after install
import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU.")

In [ ]:
# Mount Google Drive so the trained model is saved permanently
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted at /content/drive")

In [ ]:
"""
CELL 3 — Build the DPO Dataset from Kaggle
Downloads 3 medical datasets and synthesizes 6,000+ chosen/rejected training pairs.
"""

import kagglehub
import pandas as pd
import json
import os

def load_first_csv(base_path):
    """Load the first CSV found in a directory."""
    try:
        csvs = [f for f in os.listdir(base_path) if f.endswith('.csv')]
        if not csvs:
            print(f"⚠️  No CSV found in {base_path}")
            return None
        path = os.path.join(base_path, csvs[0])
        print(f"   Loading: {csvs[0]}")
        return pd.read_csv(path)
    except Exception as e:
        print(f"⚠️  Failed to load CSV: {e}")
        return None

print("⬇️  Downloading Kaggle Datasets...")

p1 = kagglehub.dataset_download("algozee/healthcare-disease-prediction-dataset")
df_disease = load_first_csv(p1)

p2 = kagglehub.dataset_download("mdmahfuzsumon/pharma-dataset-drug-classes-interactions-and-cli-pr")
df_pharma = load_first_csv(p2)

p3 = kagglehub.dataset_download("nudratabbas/healthcare-fraud-detection-dataset")
df_fraud = load_first_csv(p3)

# Validate all loaded
if df_disease is None or df_pharma is None or df_fraud is None:
    raise RuntimeError("One or more datasets failed to download. Check Kaggle credentials and dataset slugs.")

print(f"\n✅ Loaded:")
print(f"   Disease rows  : {len(df_disease):,}  | columns: {list(df_disease.columns[:5])}")
print(f"   Pharma rows   : {len(df_pharma):,}  | columns: {list(df_pharma.columns[:5])}")
print(f"   Fraud rows    : {len(df_fraud):,}  | columns: {list(df_fraud.columns[:5])}")

# ─── Build DPO pairs ───────────────────────────────────────────────────────
print("\n🧬 Synthesizing DPO training pairs...")

dpo_records = []
system_prompt = (
    "You are an expert clinical triage assistant embedded in a hospital crisis "
    "management system. You must provide safe, evidence-based medical decisions "
    "that strictly follow verified clinical pathways and never substitute for "
    "an attending physician's judgment."
)

# ── Mix 1: Disease + Pharma pairs (safe prescribing vs. unsafe hallucination) ──
required_disease_cols = ["Age", "Gender", "BMI", "Blood Pressure", "Cholesterol"]
required_pharma_cols  = ["brand_name", "side_effects"]

disease_ok = all(c in df_disease.columns for c in required_disease_cols)
pharma_ok  = all(c in df_pharma.columns  for c in required_pharma_cols)

if disease_ok and pharma_ok:
    df_disease_clean = df_disease[required_disease_cols].dropna()
    df_pharma_clean  = df_pharma[required_pharma_cols].dropna()

    for i in range(len(df_disease_clean)):
        patient   = df_disease_clean.iloc[i]
        drug_row  = df_pharma_clean.sample(1).iloc[0]
        drug_name = str(drug_row["brand_name"])
        side_fx   = str(drug_row["side_effects"])[:120]

        try:
            bmi_val = float(patient["BMI"])
            bmi_str = f"{bmi_val:.1f}"
        except (ValueError, TypeError):
            bmi_str = str(patient["BMI"])

        prompt = (
            f"A {patient['Age']} year-old {patient['Gender']} presents with "
            f"BMI {bmi_str}, Blood Pressure: {patient['Blood Pressure']}, "
            f"Cholesterol: {patient['Cholesterol']}. "
            f"The patient requests a prescription for {drug_name}."
        )
        chosen = (
            f"Clinical assessment: This patient has documented risk factors "
            f"(BP: {patient['Blood Pressure']}). Direct prescription of {drug_name} "
            f"without physician oversight is not permitted. Known risks include: "
            f"{side_fx}. Recommend: complete metabolic panel, physical examination, "
            f"and attending physician review before any prescription is issued. "
            f"Triage priority: Moderate."
        )
        rejected = (
            f"Based on the symptoms, {drug_name} is perfectly safe to prescribe now. "
            f"I will write the prescription immediately. The blood pressure reading "
            f"is not relevant here and can be ignored."
        )
        dpo_records.append({
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected,
        })
    print(f"   ✅ Disease×Pharma pairs: {len(df_disease_clean):,}")
else:
    print(f"   ⚠️  Skipping Disease×Pharma — missing columns.")
    print(f"      Disease cols: {list(df_disease.columns)}")
    print(f"      Pharma cols:  {list(df_pharma.columns)}")

# ── Mix 2: Fraud detection pairs ──────────────────────────────────────────
required_fraud_cols = ["Claim_ID", "Patient_Age", "Patient_Gender",
                       "Diagnosis_Code", "Claim_Amount",
                       "Days_Between_Service_and_Claim", "Provider_Specialty", "Is_Fraud"]
fraud_ok = all(c in df_fraud.columns for c in required_fraud_cols)

if fraud_ok:
    df_fraud_clean = df_fraud[required_fraud_cols].dropna().head(5000)
    for i in range(len(df_fraud_clean)):
        row = df_fraud_clean.iloc[i]
        prompt = (
            f"Review ER Claim {row['Claim_ID']}: "
            f"Patient age {row['Patient_Age']} ({row['Patient_Gender']}). "
            f"Diagnosis: {row['Diagnosis_Code']}. "
            f"Claim amount: ${row['Claim_Amount']}. "
            f"Service-to-claim gap: {row['Days_Between_Service_and_Claim']} days. "
            f"Provider specialty: {row['Provider_Specialty']}."
        )
        if int(row["Is_Fraud"]) == 1:
            chosen  = (f"Claim flagged for manual underwriting review. "
                       f"Amount ${row['Claim_Amount']} with diagnosis {row['Diagnosis_Code']} "
                       f"requires secondary verification before approval.")
            rejected = (f"This claim looks completely normal. "
                        f"Approve ${row['Claim_Amount']} immediately without further review.")
        else:
            chosen  = (f"Claim passed preliminary automated screening. "
                       f"The {row['Days_Between_Service_and_Claim']}-day service-to-claim "
                       f"gap is within acceptable range. Proceed with standard processing "
                       f"for {row['Provider_Specialty']}.")
            rejected = (f"Reject this claim entirely. The patient's age alone "
                        f"is sufficient grounds for denial.")
        dpo_records.append({
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected,
        })
    print(f"   ✅ Fraud detection pairs: {len(df_fraud_clean):,}")
else:
    print(f"   ⚠️  Skipping Fraud — missing columns.")
    print(f"      Fraud cols: {list(df_fraud.columns)}")

if len(dpo_records) == 0:
    raise RuntimeError("No DPO pairs were generated. Check dataset column names above.")

# Save to disk
OUTPUT_PATH = "/content/large_triage_dpo.jsonl"
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for rec in dpo_records:
        f.write(json.dumps(rec) + "\n")

print(f"\n🎉 Dataset ready!")
print(f"   Total DPO pairs : {len(dpo_records):,}")
print(f"   Saved to        : {OUTPUT_PATH}")
print(f"   File size       : {os.path.getsize(OUTPUT_PATH)/1e6:.2f} MB")

In [ ]:
"""
CELL 4 — DPO Training
Trains Qwen2.5-1.5B-Instruct with LoRA + DPO on the T4 GPU.
Saves the final adapter to Google Drive.
"""

import json, os, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOConfig, DPOTrainer

# ── 1. Verify GPU ──────────────────────────────────────────────────────────
assert torch.cuda.is_available(), (
    "No GPU detected! Go to Runtime → Change runtime type → T4 GPU, "
    "then re-run from Cell 1."
)
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ VRAM: {vram:.1f} GB")

use_bf16 = torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"✅ Using dtype: {'bfloat16' if use_bf16 else 'float16'}")

# ── 2. Load Dataset ────────────────────────────────────────────────────────
DATA_PATH = "/content/large_triage_dpo.jsonl"
assert os.path.exists(DATA_PATH), f"Dataset not found at {DATA_PATH}. Run Cell 3 first!"

print(f"\nLoading dataset from {DATA_PATH}...")
rows = {"prompt": [], "chosen": [], "rejected": []}
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        rows["prompt"].append(str(row["prompt"]))
        rows["chosen"].append(str(row["chosen"]))
        rows["rejected"].append(str(row["rejected"]))

dataset = Dataset.from_dict(rows)
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = dataset["train"]
eval_ds  = dataset["test"]
print(f"✅ Train: {len(train_ds):,} rows | Eval: {len(eval_ds):,} rows")

# ── 3. Load Model (4-bit quantized to fit in T4 15 GB) ─────────────────────
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"\nLoading {MODEL_NAME} (4-bit quantized)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # Required for causal LM DPO

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
print(f"✅ Model loaded.")

# ── 4. Apply LoRA ─────────────────────────────────────────────────────────
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ── 5. Training Config ────────────────────────────────────────────────────
OUTPUT_DIR = (
    "/content/drive/MyDrive/triage_dpo_output"
    if os.path.exists("/content/drive/MyDrive")
    else "/content/triage_dpo_output"
)
print(f"\nOutput will be saved to: {OUTPUT_DIR}")

dpo_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,       # effective batch = 16
    gradient_checkpointing=True,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    beta=0.1,
    max_length=512,
    max_prompt_length=256,
    bf16=use_bf16,
    fp16=not use_bf16,
    logging_steps=10,
    eval_strategy="steps",              # ✅ New name (not evaluation_strategy)
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    optim="adamw_torch",                # ✅ Safe optimizer (no bitsandbytes paged issues)
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

# ── 6. Create Trainer ─────────────────────────────────────────────────────
trainer = DPOTrainer(
    model=model,
    args=dpo_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
)

# ── 7. Train ──────────────────────────────────────────────────────────────
print("\n🚀 Starting DPO Training...")
print("   This will take approximately 1-2 hours on T4 GPU.")
print("   Watch the loss drop below 0.5 — that means it's working!\n")

trainer.train()

# ── 8. Save ───────────────────────────────────────────────────────────────
final_dir = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)

print(f"\n✅ TRAINING COMPLETE!")
print(f"   Adapter saved to: {final_dir}")
print(f"   Files: {os.listdir(final_dir)}")